# 13. BM25 como referencia léxica

## 1. Configuración fijada antes de comparar

BM25 puntúa coincidencias entre términos de una consulta y un documento. La frecuencia del término contribuye de manera saturada y se tiene en cuenta la longitud del resumen. La puntuación no es una probabilidad de pertinencia.

| Decisión | Configuración de este experimento |
|---|---|
| Texto de los documentos | Solo `abstract` |
| Texto de consulta | Solo `consulta` del CSV revisado |
| Parámetros | `k1 = 1.2`, `b = 0.75` |
| Peso IDF | Positivo: `log(1 + (N - df + 0.5) / (df + 0.5))` |
| Preparación de texto | Unicode NFKC y `casefold()` |
| Tokens | Letras Unicode y números; signos, guiones y guion bajo separan tokens |
| Acentos | Se conservan |
| Stopwords, stemming, lematización y traducción | No se aplican en esta configuración |
| Términos repetidos en la consulta | Cada término distinto contribuye una vez |
| Empates | Puntuación descendente y `article_id` numérico ascendente |
| Puntuaciones guardadas | Las 519 por consulta, con precisión completa |
| Lista operativa | Solo documentos con puntuación positiva |

Para cada término distinto de la consulta se calcula:

$$
S(q,d)=\sum_{t\in U(q)}
\ln\left(1+\frac{N-n_t+0.5}{n_t+0.5}\right)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+bL_d/\overline L)}.
$$

`f(t,d)` es su frecuencia en el documento; `n_t`, el número de documentos que lo contienen; `L_d`, la longitud en tokens; y `U(q)`, los términos distintos de la consulta. Las longitudes se calculan exactamente sobre los 519 registros.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter_ns
from math import log1p
import hashlib
import json
import platform
import re
import sys
import unicodedata
import numpy as np
import pandas as pd

inicio_lectura = perf_counter_ns()
# Buscar la raíz aunque el kernel esté dentro de la nueva subcarpeta.
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/consultas_evaluacion_v1.csv").is_file()), None)
if REPO is None:
    raise FileNotFoundError("No se encuentran el catálogo y las consultas en las ubicaciones indicadas.")

BASE = REPO / "07_RAG"
PRUEBAS = BASE / "Pruebas_RAG"
PRUEBAS.mkdir(exist_ok=True)
RUTA_CATALOGO = REPO / "06_grafos/catalogo_articulos.csv"
RUTA_CONSULTAS = BASE / "consultas_evaluacion_v1.csv"
RUTA_LIBRETA = PRUEBAS / "13_bm25.ipynb"
RUTA_SALIDA = PRUEBAS / "bm25_resultados_v1.json"
if not RUTA_LIBRETA.is_file():
    raise FileNotFoundError("Guarda esta libreta como 07_RAG/Pruebas_RAG/13_bm25.ipynb antes de ejecutarla.")

def sha256_archivo(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()

def sha256_json(objeto):
    texto = json.dumps(objeto, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

SHA_CATALOGO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
SHA_CONSULTAS = "1e35b789207b49ebb3e1e088c342cda234c2153f7d1b50f0d05adbf8cf459f17"
assert sha256_archivo(RUTA_CATALOGO) == SHA_CATALOGO, "El catálogo cambió respecto de la versión revisada."
assert sha256_archivo(RUTA_CONSULTAS) == SHA_CONSULTAS, "Las consultas cambiaron respecto de la versión revisada."
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False)
consultas = pd.read_csv(RUTA_CONSULTAS, dtype=str, keep_default_na=False)
assert len(catalogo) == 519 and catalogo["article_id"].nunique() == 519
assert catalogo["abstract"].str.strip().ne("").all() and catalogo["abstract"].nunique() == 514
assert len(consultas) == 24 and consultas["query_id"].nunique() == 24
assert consultas["necesidad_id"].nunique() == 12
assert consultas["sha256_catalogo"].eq(SHA_CATALOGO).all()
assert consultas["estado_revision"].eq("revisada_por_asistente").all()
for necesidad, grupo in consultas.groupby("necesidad_id"):
    assert len(grupo) == 2 and set(grupo["idioma"]) == {"es", "en"}
    esperado = "desarrollo" if necesidad in {"N01", "N02", "N03", "N04"} else "comparacion"
    assert set(grupo["division"]) == {esperado}

# La huella del código ignora resultados y metadatos del notebook.
libreta = json.loads(RUTA_LIBRETA.read_text(encoding="utf-8"))
codigo = ["".join(c["source"]) for c in libreta["cells"] if c["cell_type"] == "code"]
SHA_CODIGO = sha256_json(codigo)
lectura_validacion_ms = (perf_counter_ns() - inicio_lectura) / 1_000_000
print("Entradas verificadas: 519 artículos, 24 consultas y 12 necesidades.")
print("Trabajo y salida: 07_RAG/Pruebas_RAG/")

Entradas verificadas: 519 artículos, 24 consultas y 12 necesidades.
Trabajo y salida: 07_RAG/Pruebas_RAG/


## 2. Preparar las frecuencias de palabras

Los títulos se usarán únicamente para leer los resultados en pantalla. La puntuación se obtiene del resumen. `BI-RADS`, por ejemplo, se divide en `bi` y `rads`; el mismo procedimiento se aplica a documentos y consultas.

Conservar los 519 registros significa que las estadísticas de BM25 también incluyen los cinco pares con resúmenes repetidos. Es el mismo universo documental de los recuperadores densos.

In [2]:
K1 = 1.2
B = 0.75
CALENTAMIENTOS = 1
REPETICIONES = 10
PATRON = r"[^\W_]+"
assert K1 > 0 and 0 <= B <= 1
assert CALENTAMIENTOS >= 0 and REPETICIONES >= 1

def tokenizar(texto):
    """Aplicar exactamente el mismo análisis léxico a documentos y consultas."""
    texto = unicodedata.normalize("NFKC", texto).casefold()
    return re.findall(PATRON, texto)

# Comprobar guiones, letras acentuadas y números antes de construir las frecuencias.
assert tokenizar("BI-RADS español 2026") == ["bi", "rads", "español", "2026"]

inicio_construccion = perf_counter_ns()
article_ids = catalogo["article_id"].to_numpy()
ids_numericos = catalogo["article_id"].astype("int64").to_numpy()
documentos = [tokenizar(texto) for texto in catalogo["abstract"]]
frecuencias = [Counter(tokens) for tokens in documentos]
longitudes = np.array([len(tokens) for tokens in documentos], dtype=np.float64)
assert (longitudes > 0).all(), "Hay un resumen sin tokens; revisar antes de continuar."
N = len(documentos)
longitud_media = float(longitudes.mean())

# Contar documentos que contienen el término, no sus repeticiones totales.
df = Counter()
for contador in frecuencias:
    df.update(contador.keys())
idf = {termino: log1p((N - cantidad + 0.5) / (cantidad + 0.5))
       for termino, cantidad in df.items()}
ajuste_longitud = K1 * (1 - B + B * longitudes / longitud_media)
construccion_ms = (perf_counter_ns() - inicio_construccion) / 1_000_000

print("Estadísticas léxicas construidas en memoria; no se crea otro archivo de índice.")
print("Documentos:", N, "| términos distintos:", len(df))
print("Tokens por resumen: mínimo", int(longitudes.min()), "| máximo", int(longitudes.max()))
print("Longitud media:", round(longitud_media, 2), "tokens.")

Estadísticas léxicas construidas en memoria; no se crea otro archivo de índice.
Documentos: 519 | términos distintos: 8493
Tokens por resumen: mínimo 24 | máximo 451
Longitud media: 155.45 tokens.


## 3. Puntuar y ordenar

La función devuelve una puntuación para cada artículo. Se ordena con precisión completa; el redondeo se utilizará únicamente al mostrar una tabla.

**Una puntuación cero indica que no hay coincidencias con los términos de la consulta bajo este analizador.** No es una anotación de irrelevancia. Una puntuación positiva tampoco demuestra pertinencia: puede proceder de palabras frecuentes. Por eso las métricas de calidad requieren juicios posteriores.

In [3]:
def recuperar_bm25(texto):
    """Tokenizar, puntuar los 519 documentos y ordenar; medir solo esas operaciones."""
    inicio = perf_counter_ns()
    # Ponderación binaria de términos de consulta: evitar repetir su contribución.
    tokens = list(dict.fromkeys(tokenizar(texto)))
    puntuaciones = np.zeros(N, dtype=np.float64)
    for termino in tokens:
        if termino not in idf:
            continue
        tf = np.fromiter((cont.get(termino, 0) for cont in frecuencias), dtype=np.float64, count=N)
        puntuaciones += idf[termino] * tf * (K1 + 1) / (tf + ajuste_longitud)
    # lexsort utiliza la última clave como principal.
    orden = np.lexsort((ids_numericos, -puntuaciones))
    tiempo_ms = (perf_counter_ns() - inicio) / 1_000_000
    return tokens, puntuaciones, orden, tiempo_ms

print("Función BM25 preparada. Parámetros fijos: k1=", K1, "y b=", B)

Función BM25 preparada. Parámetros fijos: k1= 1.2 y b= 0.75


## 4. Ejecutar las 24 consultas

Se hace un calentamiento y diez mediciones por consulta. Se guardan las muestras y su mediana. El tiempo de consulta incluye tokenización, cálculo de las 519
puntuaciones y ordenación. Excluye lectura, construcción del índice, tablas y JSON.

Los tiempos que obtengas al ejecutar esta libreta corresponderán a tu equipo. Se registra el entorno de ejecución. Esta libreta no mide el pico de RAM y no lo sustituye por una estimación del tamaño de los arrays. BM25 utiliza CPU.

Las consultas de comparación también se procesan para conservar sus resultados, pero la vista de comprobación mostrará únicamente N01, del grupo de desarrollo.

In [4]:
resultados = []
tiempos_consultas = {}
for fila in consultas.to_dict("records"):
    # Solo se envía la consulta revisada, nunca criterios ni artículos orientadores.
    for _ in range(CALENTAMIENTOS):
        recuperar_bm25(fila["consulta"])
    muestras_ms = []
    for _ in range(REPETICIONES):
        tokens, puntuaciones, orden, tiempo_ms = recuperar_bm25(fila["consulta"])
        muestras_ms.append(float(tiempo_ms))
    assert np.isfinite(puntuaciones).all() and (puntuaciones >= 0).all()

    filas_documentos = []
    rango = 0
    for posicion in orden:
        score = float(puntuaciones[posicion])
        if score > 0:
            rango += 1
        filas_documentos.append({"article_id": str(article_ids[posicion]),
                                 "score_bm25": score,
                                 "rank": rango if score > 0 else None})
    resultados.append({
        "query_id": fila["query_id"], "necesidad_id": fila["necesidad_id"],
        "division": fila["division"], "idioma": fila["idioma"], "consulta": fila["consulta"],
        "tokens_distintos": tokens,
        "terminos_en_vocabulario": [t for t in tokens if t in idf],
        "terminos_fuera_vocabulario": [t for t in tokens if t not in idf],
        "n_con_puntuacion_positiva": rango, "n_con_puntuacion_cero": N - rango,
        "documentos": filas_documentos,
    })
    tiempos_consultas[fila["query_id"]] = {
        "muestras_ms": muestras_ms, "mediana_ms": float(np.median(muestras_ms)),
        "minimo_ms": min(muestras_ms), "maximo_ms": max(muestras_ms),
    }

assert len(resultados) == 24 and all(len(r["documentos"]) == 519 for r in resultados)
print("Consultas procesadas:", len(resultados))
print("Puntuaciones calculadas:", sum(len(r["documentos"]) for r in resultados))
print("Listas operativas: solo coincidencias positivas; cero no equivale a irrelevante.")
print("No se han calculado métricas de pertinencia.")

Consultas procesadas: 24
Puntuaciones calculadas: 12456
Listas operativas: solo coincidencias positivas; cero no equivale a irrelevante.
No se han calculado métricas de pertinencia.


## 5. Comprobar funcionamiento con N01

Las tablas muestran hasta diez coincidencias positivas de la misma necesidad en español e inglés. Los títulos se añaden después de puntuar, únicamente para identificar los artículos. No se usan para modificar las consultas ni los parámetros. Esta vista no demuestra calidad ni sustituye la anotación.

Se conservan en el JSON todos los documentos, no solo los diez de la vista. Si no hay coincidencias positivas, la lista operativa queda vacía; no se rellena con artículos de puntuación cero.

In [5]:
def mostrar_resultados(query_id, k=10):
    resultado = next(r for r in resultados if r["query_id"] == query_id)
    print(query_id, "—", resultado["consulta"])
    print("Términos presentes:", resultado["terminos_en_vocabulario"])
    print("Términos ausentes:", resultado["terminos_fuera_vocabulario"])
    print("Artículos con coincidencia:", resultado["n_con_puntuacion_positiva"])
    positivos = [d for d in resultado["documentos"] if d["rank"] is not None][:k]
    if not positivos:
        print("No hay coincidencias léxicas positivas.\n")
        return
    tabla = pd.DataFrame(positivos)
    tabla["titulo"] = tabla["article_id"].map(catalogo.set_index("article_id")["titulo"])
    tabla["score_bm25"] = tabla["score_bm25"].round(6)  # Solo presentación.
    with pd.option_context("display.max_colwidth", 110):
        display(tabla[["rank", "article_id", "score_bm25", "titulo"]])
    print()

mostrar_resultados("N01-ES")
mostrar_resultados("N01-EN")

N01-ES — ¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?
Términos presentes: ['de', 'en', 'natural', 'y', 'la', 'material']
Términos ausentes: ['qué', 'trabajos', 'permiten', 'consultar', 'documentación', 'técnica', 'un', 'sistema', 'mediante', 'preguntas', 'lenguaje', 'fundamentan', 'respuesta', 'recuperado']
Artículos con coincidencia: 58


,rank,article_id,score_bm25,titulo
0,1,459,10.977224,University membership through immersive virtual environments and a sense of belongingness
1,2,117,10.064423,A retrieval-augmented-generation pipeline to help users query system-provided documentation
2,3,419,9.549432,Generating a database for deep learning-based applications in multimodal tomography of the human brain
3,4,189,9.386569,Identification of horizontal gene transference by means of anomaly detection and natural language-inspired...
4,5,507,6.360795,Recent Advances on Principles of Concurrent Data Structures
5,6,200,5.976143,La Leaderboard: A Large Language Model Leaderboard for Spanish Varieties and Languages of Spain and Latin ...
6,7,201,5.976143,La Leaderboard: A Large Language Model Leaderboard for Spanish Varieties and Languages of Spain and Latin ...
7,8,323,5.729396,"Students’ Ethical, Privacy, Design, and Cultural Perspectives on Visualizing Cognitive-Affective States in..."
8,9,96,5.413522,Unveiling with Density Functional Theory the Optical Property Variations of Three Kinds of Graphene for Ac...
9,10,184,5.314208,GIL-IIMAS UNAM at SemEval-2025 Task 3: MeSSI: A Multimodule System to Detect Hallucinated Segments in Triv...



N01-EN — Which studies support natural-language questions about system documentation and ground the answers in retrieved reference material?
Términos presentes: ['which', 'studies', 'support', 'natural', 'language', 'questions', 'about', 'system', 'documentation', 'and', 'ground', 'the', 'answers', 'in', 'retrieved', 'reference', 'material']
Términos ausentes: []
Artículos con coincidencia: 519


,rank,article_id,score_bm25,titulo
0,1,117,48.222509,A retrieval-augmented-generation pipeline to help users query system-provided documentation
1,2,184,18.479299,GIL-IIMAS UNAM at SemEval-2025 Task 3: MeSSI: A Multimodule System to Detect Hallucinated Segments in Triv...
2,3,189,12.752674,Identification of horizontal gene transference by means of anomaly detection and natural language-inspired...
3,4,149,12.094238,"Complexity, Artificial Life, and Artificial Intelligence"
4,5,224,11.454266,On the Application of Natural Language Processing Tools to the Ramirez Codex
5,6,253,11.184626,Spatial Analysis of Student Feedback on Learning Management Systems in Mathematics: Leveraging GIS for Enh...
6,7,285,11.096551,A Systematic Literature Review of 10 years of Research on Program Synthesis and Natural Language Processing
7,8,146,10.587016,Comparative Analysis of Information Extraction Techniques in the Mexican Legal System
8,9,277,9.272502,Unsupervised learning and natural language processing highlight research trends in a superbug
9,10,417,9.271639,Feature-based 3D+t descriptors of hyperactivated human sperm beat patterns


## 6. Guardar un único archivo

`bm25_resultados_v1.json` contiene la configuración, las huellas de las entradas y del código, las 24 listas con sus 519 puntuaciones, los diagnósticos y los tiempos. Los ceros se conservan con rango nulo. Las puntuaciones positivas no son qrels.

In [6]:
configuracion = {
    "version_codigo": "BM25_LOCAL_V1", "variante": "BM25 con IDF positiva",
    "campo_documento": "abstract", "campo_consulta": "consulta",
    "k1": K1, "b": B,
    "idf": "ln(1 + (N - df + 0.5) / (df + 0.5))",
    "tf": "f * (k1 + 1) / (f + k1 * (1 - b + b * longitud / longitud_media))",
    "unicode": "NFKC", "mayusculas": "casefold", "regex_tokens": PATRON,
    "eliminar_acentos": False, "stopwords": False, "stemming": False,
    "lematizacion": False, "traduccion": False, "frecuencia_consulta": "binaria",
    "empates": "score completo descendente; article_id entero ascendente",
    "candidatos_operativos": "score_bm25 > 0", "rank_ceros": None,
    "profundidad_guardada": "519 puntuaciones por consulta",
    "qrels_utilizados": False, "metricas_calidad_calculadas": False,
}
fuentes = {"catalogo": {"ruta": "06_grafos/catalogo_articulos.csv", "sha256": SHA_CATALOGO},
           "consultas": {"ruta": "07_RAG/consultas_evaluacion_v1.csv", "sha256": SHA_CONSULTAS},
           "sha256_codigo": SHA_CODIGO}
firma = sha256_json({"configuracion": configuracion, "fuentes": fuentes})
huella_resultados = sha256_json(resultados)
ejecucion = {
    "fin_utc": datetime.now(timezone.utc).isoformat(),
    "entorno": {"python": sys.version.split()[0], "numpy": np.__version__,
                "pandas": pd.__version__, "sistema": platform.platform(),
                "procesador": platform.processor(), "arquitectura": platform.machine(),
                "dispositivo": "CPU", "gpu_utilizada": False, "ram_pico_bytes": None},
    "lectura_validacion_ms": float(lectura_validacion_ms),
    "construccion_lexica_ms": float(construccion_ms),
    "calentamientos_por_consulta": CALENTAMIENTOS, "repeticiones_por_consulta": REPETICIONES,
    "alcance_tiempo_consulta": "tokenizacion + puntuacion de 519 documentos + orden; sin visualizacion ni escritura",
    "latencias_consulta": tiempos_consultas,
    "ram": "No medida; no se interpreta el tamaño de arrays como RAM total o pico.",
}

# Conservar el historial solo si corresponde al mismo experimento reproducido.
historial = []
if RUTA_SALIDA.exists():
    anterior = json.loads(RUTA_SALIDA.read_text(encoding="utf-8"))
    if anterior.get("firma_experimento") != firma:
        raise ValueError("La salida existente pertenece a otra condición. Usa otra versión de salida.")
    if anterior.get("sha256_resultados") != huella_resultados:
        raise ValueError("La misma condición produjo resultados distintos. Revisar antes de sobrescribir.")
    historial = anterior["ejecuciones"]
historial.append(ejecucion)
salida = {
    "version_esquema": "BM25_RESULTADOS_V1", "firma_experimento": firma,
    "configuracion": configuracion, "fuentes": fuentes,
    "catalogo": {"n_articulos": N, "n_abstracts_distintos": 514,
                 "n_terminos": len(df), "longitud_media_tokens": longitud_media,
                 "longitud_minima_tokens": int(longitudes.min()), "longitud_maxima_tokens": int(longitudes.max())},
    "consultas": {"n_formulaciones": 24, "n_necesidades": 12,
                  "revision": "asistida_IA", "validacion_humana": "no_realizada"},
    "sha256_resultados": huella_resultados, "resultados": resultados, "ejecuciones": historial,
}
assert sha256_archivo(RUTA_CATALOGO) == SHA_CATALOGO
assert sha256_archivo(RUTA_CONSULTAS) == SHA_CONSULTAS
RUTA_SALIDA.write_text(json.dumps(salida, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print("Guardado: 07_RAG/Pruebas_RAG/" + RUTA_SALIDA.name)
print("24 consultas; 12,456 puntuaciones;", len(historial), "ejecución(es) en el historial.")
print("SHA-256 del archivo:", sha256_archivo(RUTA_SALIDA))
print("Fuentes conservadas. Archivo preparado para la comparación posterior.")

Guardado: 07_RAG/Pruebas_RAG/bm25_resultados_v1.json
24 consultas; 12,456 puntuaciones; 1 ejecución(es) en el historial.
SHA-256 del archivo: 02d563696c0ee0ece3a8d6d040b268504e12ab0bdf585a1c877a968d58e9e111
Fuentes conservadas. Archivo preparado para la comparación posterior.


## Qué significa haber terminado esta libreta

Ya existe una referencia léxica reproducible, con resultados de las mismas consultas y los mismos documentos que se usarán para los recuperadores densos.
Un score alto de BM25 no prueba pertinencia y no es comparable directamente con la similitud vectorial de un codificador ni con scores de otra consulta.